#### This file gathers and aggregates all quantitative metrics.

In [8]:
# Imports
import pandas as pd
import numpy as np
import os

# Configuration
SPARSITY_FILE = 'master_sparsity_metrics.csv'
CONSISTENCY_FILE = 'master_consistency_metrics.csv'
DEMOGRAPHICS_FILE = 'master_demographics_metrics.csv'

In [21]:
try:
    sparsity_df = pd.read_csv(SPARSITY_FILE)
    consistency_df = pd.read_csv(CONSISTENCY_FILE)
    demographics_df = pd.read_csv(DEMOGRAPHICS_FILE)
    
    demographics_df['Reliance on Demographics (%)'] = demographics_df['Reliance on Demographics (%)'].str.replace('%', '').astype(float)
    
    display(sparsity_df.head())
    display(consistency_df.head())
    display(demographics_df.head())

except FileNotFoundError as e:
    print(f"ERROR: Could not find a result file.")
    print(e)
except Exception as e:
    print(f"An error occurred during data loading or cleaning: {e}")

,Model,Gini Coefficient,HHI Score,seed,pooling_method
0,Epsilon-Greedy (SHAP) on oulad_full,0.678347,0.102598,8,AttentionMLP
1,Epsilon-Greedy (SHAP) on oulad_aggregated,0.415751,0.044020,8,AttentionMLP
2,Linear Attention on oulad_full,0.679730,0.093356,8,AttentionMLP
3,Gated Attention on oulad_full,0.787017,0.230151,8,AttentionMLP
4,Multi-Head Attention on oulad_full,0.682296,0.115222,8,AttentionMLP


,Model,Spearman's Rank (Stability),Jaccard (K=5),seed,pooling_method
0,Epsilon-Greedy (SHAP) on oulad_full,0.996907,1.0,8,AttentionMLP
1,Linear Attention on oulad_full,0.997158,1.0,8,AttentionMLP
2,Gated Attention on oulad_full,0.986293,1.0,8,AttentionMLP
3,Multi-Head Attention on oulad_full,0.998090,1.0,8,AttentionMLP
4,Epsilon-Greedy (SHAP) on oulad_aggregated,0.996485,1.0,8,AttentionMLP


,Model,Reliance on Demographics (%),seed,pooling_method
0,Epsilon-Greedy (SHAP) on oulad_full,13.3,8,AttentionMLP
1,Epsilon-Greedy (SHAP) on oulad_aggregated,21.8,8,AttentionMLP
2,Linear Attention on oulad_full,10.9,8,AttentionMLP
3,Gated Attention on oulad_full,1.6,8,AttentionMLP
4,Multi-Head Attention on oulad_full,10.9,8,AttentionMLP


In [22]:
# Calculate mean and std
agg_functions = ['mean', 'std']
    
summary_sparsity = sparsity_df.groupby('Model')[['Gini Coefficient', 'HHI Score']].agg(agg_functions)
summary_consistency = consistency_df.groupby('Model')[["Spearman's Rank (Stability)", 'Jaccard (K=5)']].agg(agg_functions)
summary_demographics = demographics_df.groupby('Model')[['Reliance on Demographics (%)']].agg(agg_functions)

summary_df_1 = pd.merge(summary_sparsity, summary_consistency, left_index=True, right_index=True)
summary_df = pd.merge(summary_df_1, summary_demographics, left_index=True, right_index=True)

# Flatten the multi-level columns into single-level names (e.g., 'Gini Coefficient_mean')
summary_df.columns = [' '.join(col).strip() for col in summary_df.columns.values]
summary_df = summary_df.reset_index()

# Create the Final Table for Display
summary_df[['Model', 'Dataset']] = summary_df['Model'].str.split(' on ', expand=True)
    
final_table_display = summary_df.sort_values(by=['Dataset', 'Model'], ascending=[False, True]).reset_index(drop=True)

display(final_table_display)

,Model,Gini Coefficient mean,Gini Coefficient std,HHI Score mean,HHI Score std,Spearman's Rank (Stability) mean,Spearman's Rank (Stability) std,Jaccard (K=5) mean,Jaccard (K=5) std,Reliance on Demographics (%) mean,Reliance on Demographics (%) std,Dataset
0,Epsilon-Greedy (SHAP),0.660360,0.040520,0.102376,0.010141,0.995194,0.002896,1.000000,0.000000,8.975,3.228390,oulad_full
1,Gated Attention,0.719830,0.104455,0.165656,0.064678,0.990721,0.003502,1.000000,0.000000,8.725,11.083734,oulad_full
2,Linear Attention,0.729315,0.048225,0.167048,0.075366,0.994120,0.005268,1.000000,0.000000,6.650,3.073001,oulad_full
3,Multi-Head Attention,0.638810,0.047717,0.101956,0.018010,0.995853,0.003697,1.000000,0.000000,11.325,3.927998,oulad_full
4,Epsilon-Greedy (SHAP),0.486688,0.048188,0.060389,0.010975,0.997173,0.000986,0.916667,0.166667,14.150,5.609813,oulad_aggregated
5,Gated Attention,0.448893,0.049559,0.052873,0.010461,0.998871,0.000596,1.000000,0.000000,15.975,6.157042,oulad_aggregated
6,Linear Attention,0.449674,0.081644,0.054349,0.014875,0.998528,0.000632,1.000000,0.000000,18.275,5.432234,oulad_aggregated
7,Multi-Head Attention,0.428203,0.048723,0.048676,0.007177,0.998033,0.000649,0.833333,0.192450,20.100,4.040627,oulad_aggregated
